In [2]:
import sys
print("python:", sys.version.split()[0])
print("kernel:", sys.executable, "\n")

for name in ["coffea", "uproot", "awkward", "numpy", "matplotlib", "hist", "mplhep", "vector"]:
    try:
        mod = __import__(name)
        print(f"  [ok]      {name:<12s} {getattr(mod, '__version__', '?')}")
    except ImportError:
        print(f"  [MISSING] {name}")

python: 3.10.14
kernel: /depot/cms/conda_envs/bhanda25/Coffea-Santosh/bin/python 

  [ok]      coffea       2024.2.2
  [ok]      uproot       5.3.10
  [ok]      awkward      2.6.5
  [ok]      numpy        1.26.4
  [ok]      matplotlib   3.9.1
  [ok]      hist         2.7.3
  [ok]      mplhep       0.3.50
  [ok]      vector       1.4.1


In [3]:
import os
import pathlib
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

# coffea 2024.2.2 warns on import that a future release will hand its four-vector
# maths to `vector`, the package you used in Lab 1 (more on that in §1.3). Nothing
# we do is affected, so we silence that one warning. Silencing warnings you have
# not read is a bad habit; this one has been read.
warnings.filterwarnings("ignore", category=FutureWarning, module="coffea.*")

import awkward as ak
import hist
import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np

import coffea
from coffea import processor
from coffea.nanoevents import NanoEventsFactory, BaseSchema
from coffea.nanoevents.methods import candidate

from utils import chunk_jobs, plot_dimuon_spectrum

hep.style.use("CMS")
print("coffea", coffea.__version__)

coffea 2024.2.2


In [4]:
import json
import urllib.request

RECID = 37728
with urllib.request.urlopen(f"https://opendata.cern.ch/api/records/{RECID}", timeout=60) as r:
    meta = json.load(r)["metadata"]

print("title      :", meta["title"])
print("DOI        :", meta["doi"])
print("licence    :", meta["license"])
print("distribution:", json.dumps(meta["distribution"], indent=2))
meta_files = []
for j in range(len(meta["_file_indices"][0]['files'])):
    #print("Files:", meta["_file_indices"][i]['files'][j])
    meta_files.append(meta["_file_indices"][0]['files'][j])
print(meta_files[0:3])

title      : /GluGluHToZZTo4L_M125_TuneCP5_13TeV_powheg2_JHUGenV7011_pythia8/RunIISummer20UL16NanoAODv9-106X_mcRun2_asymptotic_v17-v2/NANOAODSIM
DOI        : 10.7483/OPENDATA.CMS.4GPD.HQ5T
licence    : {'attribution': 'CC0-1.0'}
distribution: {
  "formats": [
    "nanoaodsim",
    "root"
  ],
  "number_events": 1000000,
  "number_files": 8,
  "size": 1696039660
}
[{'availability': 'online', 'bucket': 'b5a9ba36-e6a9-4ddb-bc0d-30f5415b0fb6', 'checksum': 'adler32:5387cdcc', 'file_id': '71753ebe-bad9-40e4-bb50-b3ed3ea9139b', 'filename': '34976A9F-8C57-044D-BB81-41704D8B017F.root', 'key': 'CMS_mc_RunIISummer20UL16NanoAODv9_GluGluHToZZTo4L_M125_TuneCP5_13TeV_powheg2_JHUGenV7011_pythia8_NANOAODSIM_106X_mcRun2_asymptotic_v17-v2_260000_file_index.json_0', 'size': 9521067, 'tags': {}, 'uri': 'root://eospublic.cern.ch//eos/opendata/cms/mc/RunIISummer20UL16NanoAODv9/GluGluHToZZTo4L_M125_TuneCP5_13TeV_powheg2_JHUGenV7011_pythia8/NANOAODSIM/106X_mcRun2_asymptotic_v17-v2/260000/34976A9F-8C57-044D-BB8

In [5]:
import os, pathlib, shutil, subprocess, time
from XRootD import client
import urllib.request

for i in range(len(meta_files)):
    XROOTD_URL = meta_files[i]["uri"]
    HTTPS_URL  = XROOTD_URL.replace("root://eospublic.cern.ch/", "https://eospublic.cern.ch/")
    EXPECTED   = meta_files[i]["size"]
    
    DATA_DIR = pathlib.Path(os.environ.get("CLUSTER_SCRATCH", pathlib.Path.home())) / "higgs_signal_data"
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_FILE = DATA_DIR / meta_files[i]["key"]
    
    if LOCAL_FILE.exists() and LOCAL_FILE.stat().st_size == EXPECTED:
        print("Already downloaded:", LOCAL_FILE)
    else:
        t0, ok = time.time(), False
    
        if shutil.which("xrdcp"):
            print("Trying xrdcp ...")
            ok = subprocess.run(["xrdcp", "-f", XROOTD_URL, str(LOCAL_FILE)]).returncode == 0
    
        if not ok:
            try:
                print("Trying XRootD python bindings ...")
                ok = client.FileSystem("root://eospublic.cern.ch").copy(
                         XROOTD_URL, str(LOCAL_FILE), force=True)[0].ok
            except ImportError:
                pass
    
        if not ok:
            print("Falling back to HTTPS ...")
            urllib.request.urlretrieve(HTTPS_URL, LOCAL_FILE)
    
        print(f"done in {time.time()-t0:.0f} s -> {LOCAL_FILE}")
    
    print(f"\nsize on disk: {LOCAL_FILE.stat().st_size:,} bytes")


Trying XRootD python bindings ...
done in 13 s -> /scratch/scholar/evans793/higgs_signal_data/CMS_mc_RunIISummer20UL16NanoAODv9_GluGluHToZZTo4L_M125_TuneCP5_13TeV_powheg2_JHUGenV7011_pythia8_NANOAODSIM_106X_mcRun2_asymptotic_v17-v2_260000_file_index.json_0

size on disk: 9,521,067 bytes
Trying XRootD python bindings ...
done in 4 s -> /scratch/scholar/evans793/higgs_signal_data/CMS_mc_RunIISummer20UL16NanoAODv9_GluGluHToZZTo4L_M125_TuneCP5_13TeV_powheg2_JHUGenV7011_pythia8_NANOAODSIM_106X_mcRun2_asymptotic_v17-v2_260000_file_index.json_1

size on disk: 64,816,412 bytes
Trying XRootD python bindings ...
done in 5 s -> /scratch/scholar/evans793/higgs_signal_data/CMS_mc_RunIISummer20UL16NanoAODv9_GluGluHToZZTo4L_M125_TuneCP5_13TeV_powheg2_JHUGenV7011_pythia8_NANOAODSIM_106X_mcRun2_asymptotic_v17-v2_260000_file_index.json_2

size on disk: 103,370,730 bytes
Trying XRootD python bindings ...
done in 6 s -> /scratch/scholar/evans793/higgs_signal_data/CMS_mc_RunIISummer20UL16NanoAODv9_GluGluHT

In [6]:
import zlib
for j in range(len(meta["_file_indices"][0]['files'])):
    # YOUR CODE HERE
    XROOTD_URL = meta_files[0]["uri"]
    HTTPS_URL  = XROOTD_URL.replace("root://eospublic.cern.ch/", "https://eospublic.cern.ch/")
    EXPECTED   = meta_files[0]["size"]
    
    DATA_DIR = pathlib.Path(os.environ.get("CLUSTER_SCRATCH", pathlib.Path.home())) / "higgs_signal_data"
    # DATA_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_FILE = DATA_DIR / meta_files[j]["key"]
              
    local_size = LOCAL_FILE.stat().st_size
    print (f"Filesize (bytes): {local_size:,}")
    print ("Filesize (MB): " + str(round(local_size/1000000, 2)))
    print("Size matches:", local_size == meta["_file_indices"][0]["files"][j]["size"])
    
    running_value = 1
    with open(LOCAL_FILE, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            running_value = zlib.adler32(chunk, running_value)
    checksum = f"{running_value:08x}"
    expected_checksum = meta["_file_indices"][0]["files"][j]["checksum"].replace("adler32:", "")
    print("Checksum matches: " + str(checksum == expected_checksum))

Filesize (bytes): 9,521,067
Filesize (MB): 9.52
Size matches: True
Checksum matches: True
Filesize (bytes): 64,816,412
Filesize (MB): 64.82
Size matches: True
Checksum matches: True
Filesize (bytes): 103,370,730
Filesize (MB): 103.37
Size matches: True
Checksum matches: True
Filesize (bytes): 160,036,442
Filesize (MB): 160.04
Size matches: True
Checksum matches: True
Filesize (bytes): 263,985,987
Filesize (MB): 263.99
Size matches: True
Checksum matches: True
Filesize (bytes): 1,024,828,691
Filesize (MB): 1024.83
Size matches: True
Checksum matches: True
Filesize (bytes): 14,635,769
Filesize (MB): 14.64
Size matches: True
Checksum matches: True
Filesize (bytes): 54,844,562
Filesize (MB): 54.84
Size matches: True
Checksum matches: True


In [7]:
import pathlib
import uproot

events = []
for j in range(len(meta["_file_indices"][0]['files'])):
    # YOUR CODE HERE
    XROOTD_URL = meta_files[0]["uri"]
    HTTPS_URL  = XROOTD_URL.replace("root://eospublic.cern.ch/", "https://eospublic.cern.ch/")
    EXPECTED   = meta_files[0]["size"]
    
    DATA_DIR = pathlib.Path(os.environ.get("CLUSTER_SCRATCH", pathlib.Path.home())) / "higgs_signal_data"
    # DATA_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_FILE = DATA_DIR / meta_files[j]["key"]
    
    try:
        FILE = str(LOCAL_FILE)                      # from Part 2
        assert pathlib.Path(FILE).exists()
        print("Using local file from Part 2")
    except (NameError, AssertionError):
        FILE = ("root://eospublic.cern.ch//eos/opendata/cms/derived-data/"
                "AOD2NanoAODOutreachTool/ForHiggsTo4Leptons/ZZTo4mu.root")
        # /eos/opendata/cms/dataset-semantics/NanoAODSIM/75589/ZZTo4L_TuneCP5_13TeV_powheg_pythia8_doc.root
        print("Local file not found - streaming over XRootD (slower)")
    
    events.append(uproot.open(FILE)["Events"])
count = 0
for i in range(len(events)):
    count += events[i].num_entries
#print(f"\nnumber of events: {(events[0].num_entries)}")
print("Number of Events:", count)

Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Using local file from Part 2
Number of Events: 1000000


In [18]:
# !!! VERY DANGEROUS ADDS A LOT OF LINES IN OUTPUT !!! #
for i in range(len(events)):
    for name, branch in events[0].items():
        print(f"  {name:<24} {branch.typename}")